# EEGNet Temporal Decoding — Rock-Paper-Scissors EEG

EEGNet-based temporal decoding to compare against the LDA baseline.

**Key design choices to match the LDA pipeline:**
- Same preprocessing (average re-reference, phase splitting, baseline correction, 250ms time bins)
- Same chunk assignment (`cosmo_chunkize`-equivalent deterministic balancing)
- Same pseudo-trial averaging (`cosmo_average_samples` with `count=4, repeats=20, seed=1`)
- Same 10-fold cross-validation using chunk labels as folds
- Same trial exclusion (first trial of each block, no-response trials)

**Only difference:** EEGNet classifier instead of regularised LDA.

In [ ]:
!pip install mne --quiet

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import mne
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

mne.set_log_level("WARNING")

In [ ]:
# ==========================================================
# PATHS — Update these for your Kaggle dataset
# ==========================================================
DATASET_PATH = "/kaggle/input/datasets/felpspotenza/eeg-dataset/eeg_trials_results"  # folder with TSV files
EEG_PATH = "/kaggle/input/datasets/felpspotenza/eeg-dataset/eeg_preprocessed"         # folder with .fif files
WORKING_PATH = "/kaggle/working/eegnet_results"

os.makedirs(WORKING_PATH, exist_ok=True)
print(f"EEG path: {EEG_PATH}")
print(f"Saving to: {WORKING_PATH}")

In [ ]:
# ==========================================================
# CONSTANTS — Matching the LDA pipeline exactly
# ==========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

PAIR_IDS = list(range(1, 10)) + list(range(11, 23)) + list(range(25, 35))
NUM_TRIALS = 480
TRIALS_PER_BLOCK = 40
NUM_CHAN = 64

# Cross-validation
N_FOLDS = 10

# Pseudo-trial parameters (matching LDA: cosmo_average_samples)
PSEUDO_COUNT = 4
PSEUDO_REPEATS = 20
PSEUDO_SEED = 1

# EEGNet training
EPOCHS = 50
PATIENCE = 10
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-3

# Time bin edges (matching LDA pipeline)
TIME_WINDOWS_AB = np.column_stack(
    [np.arange(0, 2.0, 0.25), np.arange(0.25, 2.25, 0.25)]
)  # 8 bins for Decision and Response

TIME_WINDOWS_C = np.column_stack(
    [np.arange(0, 1.0, 0.25), np.arange(0.25, 1.25, 0.25)]
)  # 4 bins for Feedback

## Helper functions — Matching LDA pipeline

In [ ]:
def cosmo_sample_unique(k, n, count, seed=None):
    """
    Balanced sampling without replacement, matching CoSMoMVPA.
    Returns (k, count) array of indices in [0, n-1].
    """
    rng = np.random.default_rng(seed)
    rs_mat = np.zeros((n, count + 1), dtype=int)
    for c in range(count + 1):
        rs_mat[:, c] = rng.permutation(n)
    rs = rs_mat.ravel(order='F')

    samples = np.zeros((k, count), dtype=int)
    visited = np.zeros(len(rs), dtype=bool)
    first_non_visited = 0

    for col in range(count):
        in_bin = np.zeros(n, dtype=bool)
        pos = first_non_visited
        for row in range(k):
            while visited[pos] or in_bin[rs[pos]]:
                pos += 1
            r = rs[pos]
            in_bin[r] = True
            samples[row, col] = r
            visited[pos] = True
        while first_non_visited < len(visited) and visited[first_non_visited]:
            first_non_visited += 1

    samples.sort(axis=0)
    return samples


def cosmo_chunkize(targets, n_chunks):
    """Deterministic balanced chunk assignment matching CoSMoMVPA."""
    n = len(targets)
    chunks = np.zeros(n, dtype=int)
    for t in np.unique(targets):
        idx = np.where(targets == t)[0]
        for i, ix in enumerate(idx):
            chunks[ix] = (i % n_chunks) + 1
    return chunks


def cosmo_average_samples(data, targets, chunks, count=4, repeats=20, seed=1):
    """Balanced pseudo-trial averaging matching CoSMoMVPA."""
    avg_data_list, avg_targets_list, avg_chunks_list = [], [], []

    for chunk_val in np.unique(chunks):
        for target_val in np.unique(targets):
            mask = (targets == target_val) & (chunks == chunk_val)
            indices = np.where(mask)[0]
            if len(indices) == 0:
                continue
            sample_ids = cosmo_sample_unique(count, len(indices), repeats, seed=seed)
            for rep in range(repeats):
                chosen = indices[sample_ids[:, rep]]
                avg_data_list.append(data[chosen].mean(axis=0))
                avg_targets_list.append(target_val)
                avg_chunks_list.append(chunk_val)

    return np.array(avg_data_list), np.array(avg_targets_list), np.array(avg_chunks_list)

In [ ]:
def epoch_to_timebinned_array(epochs):
    """
    Split epochs into Decision/Response/Feedback, baseline correct,
    and average into 250ms time bins. Matches LDA pipeline exactly.
    """
    if not epochs.preload:
        epochs.load_data()
    full_data = epochs.get_data(copy=True)
    times = epochs.times

    # Split into 3 parts
    mask_a = (times >= -0.2) & (times <= 2.0)
    times_a = times[mask_a]
    data_a = full_data[:, :, mask_a]

    mask_b = (times >= 1.8) & (times <= 4.0)
    times_b = times[mask_b]
    data_b = full_data[:, :, mask_b]

    mask_c = (times >= 3.8) & (times <= 5.0)
    times_c = times[mask_c]
    data_c = full_data[:, :, mask_c]

    # Shift time labels
    times_b_shifted = times_b - 2.0
    times_c_shifted = times_c - 4.0

    # Baseline correction [-0.2, 0]
    def baseline_correct(data_part, times_part):
        bl_mask = (times_part >= -0.2) & (times_part <= 0)
        if bl_mask.sum() > 0:
            bl_mean = data_part[:, :, bl_mask].mean(axis=2, keepdims=True)
            return data_part - bl_mean
        return data_part

    data_a = baseline_correct(data_a, times_a)
    data_b = baseline_correct(data_b, times_b_shifted)
    data_c = baseline_correct(data_c, times_c_shifted)

    # Average into time bins (strict inequalities matching MATLAB)
    def bin_data(data_part, times_part, windows):
        n_t, n_ch, _ = data_part.shape
        n_bins = windows.shape[0]
        binned = np.zeros((n_t, n_ch, n_bins))
        for w in range(n_bins):
            t_mask = (times_part > windows[w, 0]) & (times_part < windows[w, 1])
            if t_mask.sum() > 0:
                binned[:, :, w] = data_part[:, :, t_mask].mean(axis=2)
        return binned

    binned_a = bin_data(data_a, times_a, TIME_WINDOWS_AB)
    binned_b = bin_data(data_b, times_b_shifted, TIME_WINDOWS_AB)
    binned_c = bin_data(data_c, times_c_shifted, TIME_WINDOWS_C)

    data = np.concatenate([binned_a, binned_b, binned_c], axis=2)

    time_labels = np.concatenate([
        TIME_WINDOWS_AB[:, 1],
        TIME_WINDOWS_AB[:, 1] + 2.0,
        TIME_WINDOWS_C[:, 1] + 4.0,
    ])

    return data, time_labels


def build_behaviour_matrices(events_df):
    """Build behaviour matrices for Player 1 and 2, matching LDA pipeline."""
    p1_resp = events_df["player1_resp"].values
    p2_resp = events_df["player2_resp"].values
    outcome = events_df["outcome"].values
    n = len(p1_resp)

    # Player 1
    p1_prev_self = np.concatenate([[np.nan], p1_resp[:-1]])
    p1_prev_other = np.concatenate([[np.nan], p2_resp[:-1]])
    player1 = np.column_stack([p1_resp, p2_resp, outcome, p1_prev_self, p1_prev_other])

    # Player 2 (outcome recoded)
    p2_outcome = np.zeros(n, dtype=float)
    p2_outcome[outcome == 1] = 1
    p2_outcome[outcome == 2] = 3
    p2_outcome[outcome == 3] = 2
    p2_prev_self = np.concatenate([[np.nan], p2_resp[:-1]])
    p2_prev_other = np.concatenate([[np.nan], p1_resp[:-1]])
    player2 = np.column_stack([p2_resp, p1_resp, p2_outcome, p2_prev_self, p2_prev_other])

    return player1, player2


def remove_block_first_trials(data, behav):
    """Remove first trial of each block (matching MATLAB rem_idx = 1:40:480)."""
    rem_idx = np.arange(0, NUM_TRIALS, TRIALS_PER_BLOCK)
    keep_mask = np.ones(NUM_TRIALS, dtype=bool)
    keep_mask[rem_idx] = False
    return data[keep_mask], behav[keep_mask]

## EEGNet Model

In [ ]:
class EEGNet(nn.Module):
    """
    EEGNet for single-timebin classification.

    Input shape: (batch, 1, n_channels, 1)
    — one spatial map per time bin, no temporal convolution needed.

    Architecture:
      1. Spatial convolution across channels (learns spatial filters)
      2. BatchNorm + ELU + Dropout
      3. Linear classifier
    """

    def __init__(self, n_channels, n_classes, F1=8, dropout=0.5):
        super().__init__()

        self.spatial_block = nn.Sequential(
            nn.Conv2d(1, F1, (n_channels, 1), bias=False),
            nn.BatchNorm2d(F1),
            nn.ELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(F1, n_classes)

    def forward(self, x):
        # x: (batch, 1, n_channels, 1)
        x = self.spatial_block(x)   # (batch, F1, 1, 1)
        x = x.view(x.size(0), -1)  # (batch, F1)
        return self.classifier(x)   # (batch, n_classes)

## Training and evaluation

In [ ]:
def balanced_accuracy(preds, y):
    """Compute balanced accuracy (mean per-class accuracy)."""
    pred_labels = preds.argmax(dim=1)
    classes = torch.unique(y)
    accs = []
    for c in classes:
        mask = y == c
        if mask.sum() > 0:
            accs.append((pred_labels[mask] == y[mask]).float().mean())
    return torch.stack(accs).mean().item()


def train_and_evaluate_eegnet(
    X_train, y_train, X_test, y_test, n_channels, n_classes
):
    """
    Train EEGNet on a single time bin and return test accuracy.

    Uses an 80/20 train/val split from the training set for early stopping,
    then evaluates on the held-out test set.
    """
    # Shape: (n_samples, n_channels) -> (n_samples, 1, n_channels, 1)
    Xtr = torch.tensor(X_train[:, None, :, None], dtype=torch.float32)
    Xte = torch.tensor(X_test[:, None, :, None], dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.long)
    yte = torch.tensor(y_test, dtype=torch.long)

    # Train/val split
    n_train = int(0.8 * len(Xtr))
    X_tr_final = Xtr[:n_train].to(DEVICE)
    y_tr_final = ytr[:n_train].to(DEVICE)
    X_val = Xtr[n_train:].to(DEVICE)
    y_val = ytr[n_train:].to(DEVICE)
    Xte = Xte.to(DEVICE)
    yte = yte.to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(X_tr_final, y_tr_final),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    model = EEGNet(n_channels, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = None
    counter = 0

    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val)
            val_acc = balanced_accuracy(val_preds, y_val)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1

        if counter >= PATIENCE:
            break

    # Evaluate on test set
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_preds = model(Xte)
        test_acc = balanced_accuracy(test_preds, yte)

    return test_acc

In [ ]:
def temporal_decoding_eegnet(avg_data, avg_targets, avg_chunks):
    """
    Time-resolved EEGNet decoding using the same 10-fold CV
    structure as the LDA pipeline (chunk-based folds).

    Parameters
    ----------
    avg_data : (n_pseudo, n_channels, n_timebins)
    avg_targets : (n_pseudo,) class labels (1, 2, 3)
    avg_chunks : (n_pseudo,) fold labels (1..10)

    Returns
    -------
    acc : (n_timebins,) decoding accuracy per time bin
    """
    n_timebins = avg_data.shape[2]
    unique_chunks = np.unique(avg_chunks)
    n_classes = len(np.unique(avg_targets))
    n_channels = avg_data.shape[1]

    # Convert targets to 0-indexed for PyTorch CrossEntropyLoss
    classes = np.unique(avg_targets)
    target_map = {c: i for i, c in enumerate(classes)}
    targets_0idx = np.array([target_map[t] for t in avg_targets])

    acc = np.zeros(n_timebins)

    for t in range(n_timebins):
        fold_accs = []

        for test_chunk in unique_chunks:
            test_mask = avg_chunks == test_chunk
            train_mask = ~test_mask

            X_train = avg_data[train_mask, :, t]
            y_train = targets_0idx[train_mask]
            X_test = avg_data[test_mask, :, t]
            y_test = targets_0idx[test_mask]

            fold_acc = train_and_evaluate_eegnet(
                X_train, y_train, X_test, y_test, n_channels, n_classes
            )
            fold_accs.append(fold_acc)

        acc[t] = np.mean(fold_accs)

    return acc

## Main pipeline

In [ ]:
def run_eegnet_pipeline():
    """
    Main loop: iterate over pairs and players, matching the LDA pipeline
    structure exactly, but using EEGNet instead of LDA.
    """
    all_decoding = {t: [] for t in range(4)}
    target_columns = [0, 1, 3, 4]  # self, other, self_prev, other_prev
    target_names = ["self", "other", "self_prev", "other_prev"]

    for p_idx, pair in enumerate(PAIR_IDS):
        print(f"\nPair {p_idx + 1}/{len(PAIR_IDS)} (ID {pair:02d})")

        # Load events
        events_path = os.path.join(
            DATASET_PATH, f"sub-{pair:02d}_task-RPS_events.tsv"
        )
        if not os.path.exists(events_path):
            print(f"  Events not found, skipping")
            continue
        events_df = pd.read_csv(events_path, sep="\t")
        player1_behav, player2_behav = build_behaviour_matrices(events_df)
        all_behav = [player1_behav, player2_behav]

        for ppt in range(2):
            player_num = ppt + 1
            print(f"  Player {player_num}")

            epo_path = os.path.join(
                EEG_PATH, f"pair-{pair:02d}_player-{player_num}_task-RPS_eeg_epo.fif"
            )
            if not os.path.exists(epo_path):
                print(f"    Skipping: file not found")
                continue

            # Load and re-reference
            epochs = mne.read_epochs(epo_path, preload=True, verbose=False)
            epochs.set_eeg_reference("average", projection=False, verbose=False)

            # Time-bin the data (matching LDA)
            eeg_data, time_labels = epoch_to_timebinned_array(epochs)

            # Get behaviour
            behav = all_behav[ppt].copy()

            # Remove first trial of each block
            eeg_data, behav = remove_block_first_trials(eeg_data, behav)

            # Decode each target
            for test_idx in range(4):
                col = target_columns[test_idx]
                targets = behav[:, col].copy()

                # Remove no-responses and NaN
                valid_mask = (~np.isnan(targets)) & (targets > 0)
                ds_data = eeg_data[valid_mask]
                ds_targets = targets[valid_mask].astype(int)

                # Chunk assignment (matching LDA)
                chunks = cosmo_chunkize(ds_targets, n_chunks=N_FOLDS)

                # Pseudo-trial averaging (matching LDA)
                avg_data, avg_targets, avg_chunks = cosmo_average_samples(
                    ds_data, ds_targets, chunks,
                    count=PSEUDO_COUNT, repeats=PSEUDO_REPEATS, seed=PSEUDO_SEED,
                )

                # EEGNet temporal decoding
                print(f"    Decoding {target_names[test_idx]}...")
                temp_acc = temporal_decoding_eegnet(avg_data, avg_targets, avg_chunks)

                print(f"      Mean acc = {temp_acc.mean() * 100:.1f}%")

                all_decoding[test_idx].append({
                    "pair": pair,
                    "player": player_num,
                    "accuracy": temp_acc,
                })

            # Save per-player
            out_path = os.path.join(
                WORKING_PATH,
                f"pair-{pair:02d}_player-{player_num}_task-RPS_eegnet_decoding.npz",
            )
            save_dict = {}
            for t in range(4):
                if all_decoding[t] and all_decoding[t][-1]["pair"] == pair and all_decoding[t][-1]["player"] == player_num:
                    save_dict[f"decoding_acc_{t}"] = all_decoding[t][-1]["accuracy"]
            save_dict["time_labels"] = time_labels
            save_dict["pair"] = pair
            save_dict["player"] = player_num
            np.savez_compressed(out_path, **save_dict)
            print(f"    Saved -> {os.path.basename(out_path)}")

            del epochs, eeg_data
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Save group summary
    print("\nSaving group summary...")
    group = {}
    for t in range(4):
        if all_decoding[t]:
            group[f"decoding_{t}"] = np.stack([d["accuracy"] for d in all_decoding[t]])
            group[f"decoding_{t}_pairs"] = np.array([d["pair"] for d in all_decoding[t]])
            group[f"decoding_{t}_players"] = np.array([d["player"] for d in all_decoding[t]])
    group["time_labels"] = time_labels
    np.savez_compressed(os.path.join(WORKING_PATH, "group_eegnet_decoding_results.npz"), **group)
    print("Done.")

In [ ]:
run_eegnet_pipeline()